# Training Dynamics

Clean execution notebook for the checkpoint-dynamics and base-versus-instruction-tuning paper figures. Plot-ready summaries are fingerprint-cached under `.analysis_cache/training_dynamics`; set `FORCE_REBUILD_TRAINING_CACHE = True` after changing source aggregation.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import (
    BASE_INSTRUCT_DATA_SOURCES,
    BASE_INSTRUCT_FAMILIES,
    add_normalized_layers,
    discover_eval_runs,
    load_openended_prompt_metrics,
    load_openended_run_manifest,
    openended_method_label_from_config,
    summarize_base_instruct_methods,
    synthetic_model_display_name,
)
from vis import (
    plot_base_vs_instruct_method_grid,
    plot_training_dynamics_layer_grid,
    save_matplotlib_figure_bundle,
    set_matplotlib_paper_font,
)

set_matplotlib_paper_font()

LOG_ROOT = REPO_ROOT / "logs" / "evals"
FIG_DIR = REPO_ROOT / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PAPER_MAIN_FIG_DIR = FIG_DIR / "paper" / "main"
PAPER_APPENDIX_FIG_DIR = FIG_DIR / "paper" / "appendix"

USE_TRAINING_CACHE = True
FORCE_REBUILD_TRAINING_CACHE = False
TRAINING_CACHE_SCHEMA_VERSION = 1
TRAINING_CACHE_DIR = REPO_ROOT / ".analysis_cache" / "training_dynamics"
TRAINING_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def training_cache_path(prefix, manifest, *, settings):
    fingerprint_cols = [
        col
        for col in ("exp_id", "run_mtime_ns", "prompt_metrics_path", "model_name", "revision", "data_source", "method_label")
        if col in manifest.columns
    ]
    records = (
        manifest[fingerprint_cols]
        .sort_values(fingerprint_cols, kind="stable")
        .where(pd.notna(manifest[fingerprint_cols]), None)
        .to_dict(orient="records")
        if not manifest.empty
        else []
    )
    payload = {
        "schema_version": TRAINING_CACHE_SCHEMA_VERSION,
        "settings": settings,
        "runs": records,
    }
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:16]
    return TRAINING_CACHE_DIR / f"{prefix}_{digest}.parquet", payload


def load_or_build_training_cache(prefix, manifest, *, settings, builder):
    cache_path, payload = training_cache_path(prefix, manifest, settings=settings)
    if USE_TRAINING_CACHE and cache_path.exists() and not FORCE_REBUILD_TRAINING_CACHE:
        print(f"Loading training analysis cache: {cache_path.name}", flush=True)
        return pd.read_parquet(cache_path), cache_path, True
    print(f"Building training analysis cache: {cache_path.name}", flush=True)
    frame = builder()
    if USE_TRAINING_CACHE:
        frame.to_parquet(cache_path, index=False)
        cache_path.with_suffix(".json").write_text(
            json.dumps(payload, indent=2, sort_keys=True, default=str),
            encoding="utf-8",
        )
        print(f"Wrote training analysis cache: {cache_path}", flush=True)
    return frame, cache_path, False


## Base vs. Instruction-Tuned Models

Cached across-layer comparison of base and instruction-tuned variants using Repr-GMM, raw LogitLens top-$p$, and their dominant-language agreement.


In [ ]:
BASE_INSTRUCT_CACHE_DATA_SOURCES = tuple(BASE_INSTRUCT_DATA_SOURCES)
BASE_INSTRUCT_FIGURE_DATA_SOURCES = ("pud21", "include_10lang_3domain_cap30")
BASE_INSTRUCT_FIGURE_METHODS = ("repr", "raw-rtopp")
BASE_INSTRUCT_OUTPUTS = {
    "pud21": "14_base_vs_instruct_pud",
    "include_10lang_3domain_cap30": "15_base_vs_instruct_include",
}
BASE_INSTRUCT_PROMPT_METRIC_COLUMNS = [
    "prompt_idx",
    "prompt_id",
    "layer",
    "dominant_lang",
    "pivot",
    "entropy",
]

base_instruct_manifest = load_openended_run_manifest(
    LOG_ROOT,
    data_sources=BASE_INSTRUCT_CACHE_DATA_SOURCES,
    methods=BASE_INSTRUCT_FIGURE_METHODS,
)


def build_base_instruct_summary():
    """Build the cached per-layer base/instruct method summary."""
    prompt_df = load_openended_prompt_metrics(
        base_instruct_manifest,
        columns=BASE_INSTRUCT_PROMPT_METRIC_COLUMNS,
        add_domain_columns=False,
        show_progress=True,
    )
    prompt_df = add_normalized_layers(prompt_df, group_col="exp_id")
    return summarize_base_instruct_methods(
        prompt_df,
        data_sources=BASE_INSTRUCT_CACHE_DATA_SOURCES,
        families=BASE_INSTRUCT_FAMILIES,
        methods=BASE_INSTRUCT_FIGURE_METHODS,
        metrics=("pivot", "entropy"),
        agreement_methods=("repr", "raw-rtopp"),
        layer_cols=("layer", "layer_norm"),
    )


base_instruct_summary, base_instruct_cache_path, base_instruct_cache_hit = load_or_build_training_cache(
    "base_instruct_layer_summary",
    base_instruct_manifest,
    settings={
        "data_sources": BASE_INSTRUCT_CACHE_DATA_SOURCES,
        "families": tuple(BASE_INSTRUCT_FAMILIES),
        "methods": BASE_INSTRUCT_FIGURE_METHODS,
        "metrics": ("pivot", "entropy", "agreement"),
        "layer_cols": ("layer", "layer_norm"),
    },
    builder=build_base_instruct_summary,
)
print(f"Base/instruct summary rows: {len(base_instruct_summary):,}; cache_hit={base_instruct_cache_hit}")

for data_source in BASE_INSTRUCT_FIGURE_DATA_SOURCES:
    fig = plot_base_vs_instruct_method_grid(
        base_instruct_summary,
        data_source=data_source,
        families=BASE_INSTRUCT_FAMILIES,
        layer_col="layer",
        title=None,
    )
    save_matplotlib_figure_bundle(fig, PAPER_APPENDIX_FIG_DIR / BASE_INSTRUCT_OUTPUTS[data_source])
    plt.show()


## Checkpoint Dynamics Across Layers

Rows are checkpointed model families, columns are metrics, and each curve is one token-aligned checkpoint colored by training tokens. We make two separate plots with the same aggregation: one from representation posteriors (`repr`) and one from raw-logitlens argmax decoding (`raw-rmax`).


In [ ]:
CHECKPOINT_DATA_SOURCE = "pud21"
CHECKPOINT_MODEL_FAMILIES = {
    "swiss-ai/Apertus-8B-2509": "Apertus-8B",
    "allenai/OLMo-2-1124-7B": "OLMo-2-7B",
}
CHECKPOINT_METHODS = ("repr", "raw-rmax")
CHECKPOINT_LOAD_METHODS = ("repr", "raw-rmax", "raw-rtopp")
CHECKPOINT_METHOD_TITLES = {
    "repr": "GMM representation",
    "raw-rmax": "Raw logit-lens argmax decoding",
}
CHECKPOINT_METHOD_OUTPUTS = {
    "repr": "12_checkpoint_dynamics_repr",
    "raw-rmax": "13_checkpoint_dynamics_decoding",
}
CHECKPOINT_METRICS = ("pivot", "dominance", "entropy")
CHECKPOINT_ALIGNMENT_PATH = REPO_ROOT / "checkpoint_lists" / "token_aligned_10" / "alignment.tsv"
CHECKPOINT_MODEL_REVISION_COLUMNS = {
    "swiss-ai/Apertus-8B-2509": (
        "swiss-ai_Apertus-8B-2509_revision",
        "swiss-ai_Apertus-8B-2509_tokens",
    ),
    "allenai/OLMo-2-1124-7B": (
        "allenai_OLMo-2-1124-7B_revision",
        "allenai_OLMo-2-1124-7B_tokens",
    ),
}


In [ ]:
def _format_training_token_label(tokens):
    if pd.isna(tokens):
        return None
    tokens = float(tokens)
    if tokens >= 1_000_000_000_000:
        return f"{tokens / 1_000_000_000_000:.2g}T"
    if tokens >= 1_000_000_000:
        return f"{tokens / 1_000_000_000:.0f}B"
    return f"{tokens:.0f}"


def load_training_revision_metadata(alignment_path=CHECKPOINT_ALIGNMENT_PATH):
    """Load checkpoint revisions and their aligned training-token counts."""
    alignment = pd.read_csv(alignment_path, sep="	")
    rows = []
    for model_name, (revision_col, tokens_col) in CHECKPOINT_MODEL_REVISION_COLUMNS.items():
        for record in alignment[[revision_col, tokens_col]].dropna().drop_duplicates().itertuples(index=False):
            revision, tokens = record
            rows.append(
                {
                    "model_name": model_name,
                    "run_revision": revision,
                    "training_tokens": int(tokens),
                    "training_token_label": _format_training_token_label(tokens),
                    "training_model_family": CHECKPOINT_MODEL_FAMILIES[model_name],
                }
            )
    return pd.DataFrame(rows)


def load_training_checkpoint_manifest(log_root=LOG_ROOT):
    """Select completed checkpoint runs with known training-token counts."""
    revision_meta = load_training_revision_metadata()
    revision_lookup = revision_meta.set_index(["model_name", "run_revision"])
    manifest = discover_eval_runs(log_root, require_prompt_metrics=True)
    if manifest.empty:
        return manifest

    manifest = manifest.copy()
    manifest["data_source"] = manifest["config"].map(lambda c: c.get("data_source") if isinstance(c, dict) else None)
    manifest["method_label"] = manifest["config"].map(openended_method_label_from_config)
    manifest["decoding_decode_mode"] = manifest["config"].map(lambda c: c.get("decoding_decode_mode") if isinstance(c, dict) else None)
    manifest["include_prompt_style"] = manifest["config"].map(lambda c: c.get("include_prompt_style") if isinstance(c, dict) else None)
    manifest["pud_split_mode"] = manifest["config"].map(lambda c: c.get("pud_split_mode") if isinstance(c, dict) else None)
    manifest["display_model_name"] = manifest["model_name"].map(synthetic_model_display_name)
    manifest["run_revision"] = manifest["revision"]

    wanted_pairs = set(revision_lookup.index)
    manifest = manifest[
        manifest["data_source"].eq(CHECKPOINT_DATA_SOURCE)
        & manifest["method_label"].isin(CHECKPOINT_LOAD_METHODS)
        & manifest.apply(lambda row: (row["model_name"], row["run_revision"]) in wanted_pairs, axis=1)
    ].copy()
    if manifest.empty:
        return manifest.reset_index(drop=True)

    joined = manifest.join(revision_meta.set_index(["model_name", "run_revision"]), on=["model_name", "run_revision"])
    joined = (
        joined.sort_values(["run_mtime_ns", "exp_id"])
        .groupby(["model_name", "run_revision", "method_label"], dropna=False, as_index=False)
        .tail(1)
        .sort_values(["training_model_family", "training_tokens", "method_label"])
        .reset_index(drop=True)
    )
    return joined


def summarize_training_checkpoint_layers(prompt_df, *, method_label):
    """Average checkpoint metrics by layer for one estimator."""
    if prompt_df.empty:
        return pd.DataFrame()

    df = prompt_df[prompt_df["method_label"].eq(method_label)].copy()
    if df.empty:
        return pd.DataFrame()

    group_cols = [
        "training_model_family",
        "display_model_name",
        "model_name",
        "run_revision",
        "training_tokens",
        "training_token_label",
        "method_label",
        "layer",
    ]
    metric_frames = []
    for metric in CHECKPOINT_METRICS:
        if metric not in df.columns:
            continue
        metric_frame = (
            df.groupby(group_cols, dropna=False)[metric]
            .mean()
            .reset_index(name="value")
        )
        metric_frame["metric"] = metric
        metric_frames.append(metric_frame)

    if not metric_frames:
        return pd.DataFrame()
    return pd.concat(metric_frames, ignore_index=True)


In [ ]:
checkpoint_manifest = load_training_checkpoint_manifest(LOG_ROOT)
print(f"manifest rows: {len(checkpoint_manifest):,}")
if checkpoint_manifest.empty:
    print("No token-aligned checkpoint runs found locally yet. Sync or copy the completed eval logs into logs/evals and rerun this notebook.")
else:
    print(
        checkpoint_manifest
        .groupby(["training_model_family", "training_token_label", "method_label"], dropna=False)
        .size()
        .rename("runs")
        .reset_index()
        .pivot_table(
            index=["training_model_family", "training_token_label"],
            columns="method_label",
            values="runs",
            fill_value=0,
        )
        .to_string()
    )


In [ ]:
CHECKPOINT_PROMPT_METRIC_COLUMNS = [
    "prompt_idx",
    "prompt_id",
    "layer",
    "dominant_lang",
    "dominance",
    "entropy",
    "pivot",
]
checkpoint_prompt_df = pd.DataFrame()


def build_checkpoint_layer_summary_cache():
    """Load prompt metrics and build the checkpoint-layer summary cache."""
    global checkpoint_prompt_df
    checkpoint_prompt_df = load_openended_prompt_metrics(
        checkpoint_manifest,
        columns=CHECKPOINT_PROMPT_METRIC_COLUMNS,
        add_domain_columns=False,
        show_progress=True,
    )
    if checkpoint_prompt_df.empty:
        return pd.DataFrame()
    checkpoint_prompt_df = checkpoint_prompt_df.merge(
        checkpoint_manifest[[
            "exp_id",
            "training_model_family",
            "training_tokens",
            "training_token_label",
        ]],
        on="exp_id",
        how="left",
    )
    checkpoint_prompt_df = add_normalized_layers(checkpoint_prompt_df, group_col="exp_id")
    summaries = [
        summarize_training_checkpoint_layers(checkpoint_prompt_df, method_label=method_label)
        for method_label in CHECKPOINT_METHODS
    ]
    summaries = [summary for summary in summaries if not summary.empty]
    return pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()


checkpoint_layer_summary, checkpoint_layer_cache_path, checkpoint_layer_cache_hit = load_or_build_training_cache(
    "checkpoint_layer_summary",
    checkpoint_manifest,
    settings={
        "data_source": CHECKPOINT_DATA_SOURCE,
        "methods": tuple(CHECKPOINT_METHODS),
        "metrics": tuple(CHECKPOINT_METRICS),
    },
    builder=build_checkpoint_layer_summary_cache,
)
checkpoint_layer_summaries = {
    method_label: checkpoint_layer_summary[checkpoint_layer_summary["method_label"].eq(method_label)].copy()
    for method_label in CHECKPOINT_METHODS
}

print(f"prompt metric rows held in memory: {len(checkpoint_prompt_df):,}")
print(f"layer summary rows: {len(checkpoint_layer_summary):,}")
print(f"layer cache hit: {checkpoint_layer_cache_hit}")
for method_label, summary in checkpoint_layer_summaries.items():
    print(f"\n{method_label} layer summary rows: {len(summary):,}")
    if not summary.empty:
        print(
            summary
            .groupby(["training_model_family", "metric"], dropna=False)
            .size()
            .rename("layer_points")
            .reset_index()
            .pivot_table(
                index="training_model_family",
                columns="metric",
                values="layer_points",
                fill_value=0,
            )
            .to_string()
        )


In [ ]:
for method_label, summary in checkpoint_layer_summaries.items():
    fig = plot_training_dynamics_layer_grid(
        summary,
        model_families=("OLMo-2-7B", "Apertus-8B"),
        metrics=CHECKPOINT_METRICS,
        layer_col="layer",
        title=f"[PUD21 checkpoint dynamics: {CHECKPOINT_METHOD_TITLES[method_label]}] LLID metrics across model layers\nRows = models, Columns = metrics",
    )
    save_matplotlib_figure_bundle(
        fig,
        PAPER_APPENDIX_FIG_DIR / CHECKPOINT_METHOD_OUTPUTS[method_label],
    )
    plt.show()


## Training-Progress Dynamics by Layer Bins

Additional checkpoint-dynamics view. Rows are model families, columns expose both estimator families and their agreement, the x-axis is normalized training progress, and each line is a layer bin. Confident pivot is computed at the prompt level as `pivot AND dominance >= 0.6` for the same estimator.


In [ ]:
TRAINING_PROGRESS_DATA_SOURCE = CHECKPOINT_DATA_SOURCE
TRAINING_PROGRESS_MODEL_FAMILIES = ("OLMo-2-7B", "Apertus-8B")
TRAINING_PROGRESS_DECODE_METHOD = "auto"  # "auto" prefers raw-rtopp if present, else raw-rmax.
TRAINING_PROGRESS_CONFIDENCE_THRESHOLD = 0.6
TRAINING_PROGRESS_BIN_SCHEMES = {
    "early_middle_late": (
        "3 layer bins",
        [0.0, 0.25, 0.75, 1.000001],
        ["early", "middle", "late"],
    ),
    "quartiles": (
        "4 layer quartiles",
        [0.0, 0.25, 0.50, 0.75, 1.000001],
        ["Q1", "Q2", "Q3", "Q4"],
    ),
}


def select_paired_decoding_method(prompt_df, requested=TRAINING_PROGRESS_DECODE_METHOD):
    """Select a decoding method available for every representation checkpoint."""
    if requested != "auto":
        return requested
    if prompt_df.empty or "method_label" not in prompt_df.columns:
        return None

    key_cols = ["training_model_family", "model_name", "run_revision", "training_tokens"]
    repr_keys = (
        prompt_df[prompt_df["method_label"].eq("repr")][key_cols]
        .drop_duplicates()
    )
    if repr_keys.empty:
        return None
    repr_key_set = set(map(tuple, repr_keys.to_numpy()))

    for candidate in ("raw-rtopp", "raw-rmax"):
        candidate_keys = (
            prompt_df[prompt_df["method_label"].eq(candidate)][key_cols]
            .drop_duplicates()
        )
        candidate_key_set = set(map(tuple, candidate_keys.to_numpy()))
        if repr_key_set and repr_key_set.issubset(candidate_key_set):
            return candidate
    return None


def add_training_layer_bins(prompt_df, *, bin_edges, bin_labels):
    """Add normalized layer bins and within-family training progress."""
    df = prompt_df.copy()
    if "layer_norm" not in df.columns:
        df = add_normalized_layers(df, group_col="exp_id")
    df["training_layer_bin"] = pd.cut(
        df["layer_norm"].clip(lower=0.0, upper=1.0),
        bins=bin_edges,
        labels=bin_labels,
        include_lowest=True,
        right=False,
    )
    df["training_layer_bin"] = df["training_layer_bin"].astype(str)
    df["training_layer_bin_order"] = df["training_layer_bin"].map({label: idx for idx, label in enumerate(bin_labels)})
    max_tokens = df.groupby("training_model_family")["training_tokens"].transform("max")
    df["training_progress"] = pd.to_numeric(df["training_tokens"], errors="coerce") / max_tokens
    return df


def summarize_training_progress(
    prompt_df,
    *,
    decode_method="auto",
    bin_edges,
    bin_labels,
    confidence_threshold=TRAINING_PROGRESS_CONFIDENCE_THRESHOLD,
):
    if prompt_df.empty:
        return pd.DataFrame(), None

    selected_decode_method = select_paired_decoding_method(prompt_df, requested=decode_method)
    if selected_decode_method is None:
        return pd.DataFrame(), None

    df = add_training_layer_bins(prompt_df, bin_edges=bin_edges, bin_labels=bin_labels)
    df = df[
        df["training_model_family"].isin(TRAINING_PROGRESS_MODEL_FAMILIES)
        & df["method_label"].isin(["repr", selected_decode_method])
        & df["training_layer_bin"].isin(bin_labels)
    ].copy()
    if df.empty:
        return pd.DataFrame(), selected_decode_method

    group_cols = [
        "training_model_family",
        "training_tokens",
        "training_token_label",
        "training_progress",
        "training_layer_bin",
        "training_layer_bin_order",
    ]
    metric_frames = []

    for method_label, metric_name in [
        ("repr", "repr_dominance"),
        (selected_decode_method, "dec_dominance"),
    ]:
        method_df = df[df["method_label"].eq(method_label)].copy()
        if method_df.empty or "dominance" not in method_df.columns:
            continue
        summary = method_df.groupby(group_cols, dropna=False)["dominance"].mean().reset_index(name="value")
        summary["metric"] = metric_name
        metric_frames.append(summary)

    for method_label, metric_name in [
        ("repr", "repr_confident_pivot_rate"),
        (selected_decode_method, "dec_confident_pivot_rate"),
    ]:
        method_df = df[df["method_label"].eq(method_label)].copy()
        if method_df.empty or not {"pivot", "dominance"}.issubset(method_df.columns):
            continue
        method_df["confident_pivot"] = method_df["pivot"].astype(bool) & (
            pd.to_numeric(method_df["dominance"], errors="coerce") >= confidence_threshold
        )
        summary = method_df.groupby(group_cols, dropna=False)["confident_pivot"].mean().reset_index(name="value")
        summary["metric"] = metric_name
        metric_frames.append(summary)

    join_keys = [
        "training_model_family",
        "model_name",
        "run_revision",
        "training_tokens",
        "training_token_label",
        "training_progress",
        "layer",
        "training_layer_bin",
        "training_layer_bin_order",
        "prompt_idx",
    ]
    if "prompt_id" in df.columns:
        join_keys.append("prompt_id")
    agreement_df = df[df["method_label"].isin(["repr", selected_decode_method])].copy()
    if not agreement_df.empty and {"dominant_lang", "method_label"}.issubset(agreement_df.columns):
        wide = (
            agreement_df[join_keys + ["method_label", "dominant_lang"]]
            .drop_duplicates()
            .pivot_table(index=join_keys, columns="method_label", values="dominant_lang", aggfunc="first")
            .reset_index()
        )
        if "repr" in wide.columns and selected_decode_method in wide.columns:
            wide["agreement"] = (wide["repr"] == wide[selected_decode_method]).astype(float)
            summary = wide.groupby(group_cols, dropna=False)["agreement"].mean().reset_index(name="value")
            summary["metric"] = "agreement_repr_vs_dec"
            metric_frames.append(summary)

    if not metric_frames:
        return pd.DataFrame(), selected_decode_method
    out = pd.concat(metric_frames, ignore_index=True)
    out["decode_method"] = selected_decode_method
    return out.reset_index(drop=True), selected_decode_method


training_progress_decode_method = select_paired_decoding_method(checkpoint_manifest, requested=TRAINING_PROGRESS_DECODE_METHOD)
training_progress_manifest = checkpoint_manifest[
    checkpoint_manifest["method_label"].isin(["repr", training_progress_decode_method])
].copy()


def build_training_progress_cache():
    """Build training-progress summaries for every configured layer scheme."""
    global checkpoint_prompt_df
    if checkpoint_prompt_df.empty:
        checkpoint_prompt_df = load_openended_prompt_metrics(
            training_progress_manifest,
            columns=CHECKPOINT_PROMPT_METRIC_COLUMNS,
            add_domain_columns=False,
            show_progress=True,
        )
        checkpoint_prompt_df = checkpoint_prompt_df.merge(
            training_progress_manifest[[
                "exp_id",
                "training_model_family",
                "training_tokens",
                "training_token_label",
            ]],
            on="exp_id",
            how="left",
        )
        checkpoint_prompt_df = add_normalized_layers(checkpoint_prompt_df, group_col="exp_id")
    summaries = []
    for scheme_name, (scheme_label, bin_edges, bin_labels) in TRAINING_PROGRESS_BIN_SCHEMES.items():
        summary, decode_method = summarize_training_progress(
            checkpoint_prompt_df,
            decode_method=training_progress_decode_method,
            bin_edges=bin_edges,
            bin_labels=bin_labels,
        )
        summary["bin_scheme"] = scheme_name
        summary["bin_scheme_label"] = scheme_label
        summaries.append(summary)
    return pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()


training_progress_summary, training_progress_cache_path, training_progress_cache_hit = load_or_build_training_cache(
    "training_progress_plot_summary",
    training_progress_manifest,
    settings={
        "data_source": TRAINING_PROGRESS_DATA_SOURCE,
        "model_families": TRAINING_PROGRESS_MODEL_FAMILIES,
        "decode_method": training_progress_decode_method,
        "confidence_threshold": TRAINING_PROGRESS_CONFIDENCE_THRESHOLD,
        "bin_schemes": {
            name: {"edges": values[1], "labels": values[2]}
            for name, values in TRAINING_PROGRESS_BIN_SCHEMES.items()
        },
    },
    builder=build_training_progress_cache,
)
print(
    f"plot-summary rows: {len(training_progress_summary):,}; "
    f"decode_method={training_progress_decode_method}; cache_hit={training_progress_cache_hit}"
)


In [ ]:
# Shared renderer for the three-bin main figure and four-bin appendix figure.
TRAINING_PROGRESS_COLUMNS = (
    ("Dominance", (("repr_dominance", "repr"), ("dec_dominance", "dec"))),
    ("Confident pivot rate", (("repr_confident_pivot_rate", "repr"), ("dec_confident_pivot_rate", "dec"))),
    ("Representation-decoding agreement", (("agreement_repr_vs_dec", "agreement"),)),
)
TRAINING_PROGRESS_ESTIMATOR_STYLES = {
    "repr": {"linestyle": "-", "linewidth": 2.2},
    "dec": {"linestyle": "--", "linewidth": 2.2},
    "agreement": {"linestyle": "-", "linewidth": 2.2},
}
TRAINING_PROGRESS_BIN_COLORS = {
    "early_middle_late": {
        "early": "#0072B2",
        "middle": "#E69F00",
        "late": "#CC79A7",
    },
    "quartiles": {
        "Q1": "#0072B2",
        "Q2": "#E69F00",
        "Q3": "#009E73",
        "Q4": "#CC79A7",
    },
}
TRAINING_PROGRESS_OUTPUTS = {
    "early_middle_late": PAPER_MAIN_FIG_DIR / "06_training_dynamics",
    "quartiles": PAPER_APPENDIX_FIG_DIR / "11_training_dynamics_quartiles",
}


def plot_training_progress_overlay(
    summary,
    *,
    bin_scheme,
    bin_labels,
    decode_method,
):
    bin_colors = TRAINING_PROGRESS_BIN_COLORS[bin_scheme]
    fig, axes = plt.subplots(
        len(TRAINING_PROGRESS_MODEL_FAMILIES),
        len(TRAINING_PROGRESS_COLUMNS),
        figsize=(13.8, 6.4),
        squeeze=False,
        sharex=True,
        sharey=True,
        gridspec_kw={"hspace": 0.22, "wspace": 0.12},
    )
    for row_idx, family in enumerate(TRAINING_PROGRESS_MODEL_FAMILIES):
        for col_idx, (column_label, metric_specs) in enumerate(TRAINING_PROGRESS_COLUMNS):
            ax = axes[row_idx, col_idx]
            family_df = summary[summary["training_model_family"].eq(family)]
            for metric_name, estimator_key in metric_specs:
                metric_df = family_df[family_df["metric"].eq(metric_name)]
                style = TRAINING_PROGRESS_ESTIMATOR_STYLES[estimator_key]
                for layer_bin in bin_labels:
                    series = metric_df[metric_df["training_layer_bin"].eq(layer_bin)].sort_values("training_progress")
                    if series.empty:
                        continue
                    ax.plot(
                        series["training_progress"],
                        series["value"],
                        color=bin_colors[layer_bin],
                        marker="o",
                        markersize=3.8,
                        **style,
                    )
            ax.set_xlim(-0.02, 1.02)
            ax.set_ylim(0.0, 1.0)
            ax.grid(True, linewidth=0.45, alpha=0.25)
            ax.tick_params(labelsize=13)
            if row_idx == 0:
                ax.set_title(column_label, fontsize=15, fontweight=600, pad=9)
            if col_idx == 0:
                ax.set_ylabel(family, fontsize=15, fontweight=600, labelpad=11)
            else:
                ax.tick_params(axis="y", left=False, labelleft=False)

    layer_handles = [
        plt.Line2D([], [], color="none", linestyle="none", label="Layer bin:"),
        *[
            plt.Line2D([0], [0], color=bin_colors[layer_bin], linewidth=3.0, label=layer_bin.title())
            for layer_bin in bin_labels
        ],
    ]
    decode_legend_label = {
        "raw-rtopp": "Raw LogitLens Top-p",
        "raw-rmax": "Raw LogitLens R-max",
    }.get(decode_method, str(decode_method))
    estimator_handles = [
        plt.Line2D([], [], color="none", linestyle="none", label="Estimator:"),
        plt.Line2D([0], [0], color="#333333", linestyle="-", linewidth=2.5, label="Representation probe"),
        plt.Line2D([0], [0], color="#333333", linestyle="--", linewidth=2.5, label=decode_legend_label),
    ]
    layer_legend = fig.legend(
        handles=layer_handles,
        loc="lower center",
        bbox_to_anchor=(0.285, 0.018),
        ncol=len(layer_handles),
        frameon=False,
        fontsize=14,
        handlelength=2.0,
        columnspacing=1.15,
    )
    estimator_legend = fig.legend(
        handles=estimator_handles,
        loc="lower center",
        bbox_to_anchor=(0.79, 0.018),
        ncol=len(estimator_handles),
        frameon=False,
        fontsize=14,
        handlelength=2.0,
        columnspacing=1.25,
    )
    layer_legend.get_texts()[0].set_fontweight(600)
    estimator_legend.get_texts()[0].set_fontweight(600)

    fig.suptitle(
        "Training Dynamics of LLID Methods on PUD21",
        fontsize=18,
        fontweight=600,
        y=0.985,
    )
    fig.text(
        0.53,
        0.935,
        f"Rows = model families, columns = diagnostics; confident pivot = pivot and dominance >= {TRAINING_PROGRESS_CONFIDENCE_THRESHOLD}",
        ha="center",
        va="top",
        fontsize=13,
    )
    fig.supxlabel("Normalized training progress", fontsize=15, fontweight=600, x=0.53, y=0.115)
    fig.subplots_adjust(left=0.09, right=0.99, bottom=0.22, top=0.85)
    return fig


for scheme_name, (_, _, bin_labels) in TRAINING_PROGRESS_BIN_SCHEMES.items():
    scheme_summary = training_progress_summary[training_progress_summary["bin_scheme"].eq(scheme_name)].copy()
    if scheme_summary.empty:
        print(f"No cached summary for {scheme_name}")
        continue
    fig = plot_training_progress_overlay(
        scheme_summary,
        bin_scheme=scheme_name,
        bin_labels=bin_labels,
        decode_method=training_progress_decode_method,
    )
    save_matplotlib_figure_bundle(fig, TRAINING_PROGRESS_OUTPUTS[scheme_name])
    plt.show()
